## Statistical analysis

Specify the analysis fields to obtain a p_value for each feature using t-tests. The `group` field supports separate summaries for train, val, test, or other subsets.

Two feature types are summarized:

1. Categorical features: counts and proportions.
2. Continuous features: means and variances.

In [ ]:
import pandas as pd
import numpy as np
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.utils import print_join_info

task = get_param_in_cwd('task_column') or 'label'
p_value = get_param_in_cwd('p_value') or 0.05
# Replace with the path to your clinical data file.
test_data = pd.read_csv(get_param_in_cwd('clinic_file'))
stats_columns_settings = get_param_in_cwd('stats_columns')
continuous_columns_settings = get_param_in_cwd('continuous_columns')
mapping_columns_settings = get_param_in_cwd('mapping_columns')
test_data = test_data[[c for c in test_data.columns if c != task]]
test_data['ID'] = test_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
group_info = pd.read_csv(get_param_in_cwd('label_file'))

print_join_info(test_data, group_info)
test_data = pd.merge(test_data, group_info, on='ID', how='inner')
test_data

In [ ]:
test_data['group'].value_counts()

# Clean feature names

Remove special characters from feature names.

In [ ]:
import re

def map_cnames(x):
    x = re.split('[(|(]', x)[0]
    x = x.replace('-', '_').replace(' ', '_').replace('>', '').replace('/', '_').replace('+', '_')
    return x.strip('_')

test_data.columns = list(map(map_cnames, test_data.columns))
test_data.columns

# Analyze data

Specify feature columns for analysis, or detect them automatically if omitted.

In [ ]:
stats_columns = [c for c in stats_columns_settings or list(test_data.columns[1:-2]) if c not in ['Treatment']]
test_data = test_data.copy()[['ID'] + stats_columns + ['group', task]]
test_data#['group'].value_counts()

# Map feature categories

Specify columns requiring category mapping. Infer the range automatically when it is not provided.

In [ ]:
mapping_columns = mapping_columns_settings or [c for c in test_data.columns[1:-2] if test_data[c].dtype == object]
mapping_columns

# Map categorical data

Map nonnumeric values to categorical codes.

In [ ]:
from onekey_algo.custom.utils import map2numerical

data, mapping = map2numerical(test_data, mapping_columns=mapping_columns)
mapping

In [ ]:
data.dtypes

# Continuous feature columns

Automatically identify continuous features: columns with noninteger values or more than five distinct values are treated as continuous.

In [ ]:
from onekey_algo.custom.components.comp1 import fillna

test_data = fillna(test_data, fill_mod='50%')
continuous_columns = []
for col in test_data.columns:
    if test_data[col].apply(lambda x: x.is_integer() if isinstance(x, float) else False).all():
        test_data[col] = test_data[col].astype(int)

for c in stats_columns:
#     print(c, np.unique(test_data[c]), test_data[c].dtype)
    if len(np.unique(test_data[c])) > 8 or not np.int8 <= test_data[c].dtype <= np.int64:
        continuous_columns.append(c)
        
continuous_columns = continuous_columns_settings or continuous_columns
continuous_columns = [c for c in continuous_columns if c not in ('differentation')]
continuous_columns

# Impute missing values

In [ ]:
import os
os.makedirs('data', exist_ok=True)
data = test_data
data.to_csv('data/clinical.csv', index=False)
data

### Statistical analysis

Two output formats are supported: `pretty=True` returns a formatted table; `pretty=False` returns a dictionary.

```python
def clinic_stats(data: DataFrame, stats_columns: Union[str, List[str]], label_column='label',
                 group_column: str = None, continuous_columns: Union[str, List[str]] = None,
                 pretty: bool = True) -> Union[dict, DataFrame]:
    """

    Args:
        data: input data.
        stats_columns: columns to summarize.
        label_column: binary target column; defaults to `label`.
        group_column: subset column, such as training, test, or validation group.
        continuous_columns: continuous variables to summarize using means and variances.
        pretty: bool, whether to format the output.

    Returns:
        stats DataFrame or json

    """
```

In [ ]:
from onekey_algo.custom.components.stats import clinic_stats

pd.set_option('display.max_rows', None)
stats = clinic_stats(data, 
                     stats_columns= stats_columns,
                     label_column=task, 
                     group_column='group', 
                     continuous_columns= continuous_columns, 
                     pretty=True, verbose=False)
stats.to_csv('data/stats.csv', index=False, encoding='utf_8_sig')
stats

In [ ]:
from onekey_algo.custom.components.stats import clinic_stats

pd.set_option('display.max_rows', None)
stats_train_val = clinic_stats(data[data['group'].isin(['train', 'val'])], 
                               stats_columns= stats_columns,
                               label_column='group', 
                               group_column=None, 
                               continuous_columns= continuous_columns, 
                               pretty=True, verbose=False).reset_index(drop=True)

stats = clinic_stats(data, 
                     stats_columns= stats_columns,
                     label_column='group', 
                     group_column=None, 
                     continuous_columns= continuous_columns, 
                     pretty=True, verbose=False)
# display(stats)
stats_train_val['test'] = stats['-label=test']
stats_train_val.to_csv('data/stats_all.csv', index=False, encoding='utf_8_sig')
stats_train_val

In [ ]:
from onekey_algo.custom.components.comp1 import uni_multi_variable_analysis                        

r = uni_multi_variable_analysis(data[data['group'] == 'train'], stats_columns, save_dir='img', p_value4multi=p_value, 
                                hazard_ratios=True, label_column=task, algo='ols')

In [ ]:
uni_v = pd.read_csv('img/univariable_reg.csv')
uni_v = uni_v[uni_v['p_value'] <= 0.05]
sel_data = data[['ID'] + list(uni_v['feature_name']) + ['group', task]]
sel_data.to_csv('data/clinic_sel.csv', index=False)
sel_data

In [ ]:
uni = pd.read_csv('img/univariable_reg.csv')
uni = uni[[c for c in uni if ('OR' in c and 'Log' not in c) or c in ['feature_name', 'p_value']]]
uni['95% CI'] = [f"{y:.4f}-{z:.4f}" for x, y, z in np.array(uni[[c for c in uni.columns if c not in ['feature_name', 'p_value']]])]
multi = pd.read_csv('img/multivariable_reg.csv')
multi = multi[[c for c in uni if ('OR' in c and 'Log' not in c) or c in ['feature_name', 'p_value']]]
multi['95% CI'] = [f"{y:.4f}-{z:.4f}" for x, y, z in np.array(multi[[c for c in multi.columns if c not in ['feature_name', 'p_value']]])]
info = pd.merge(uni[['feature_name', 'OR', '95% CI', 'p_value']], multi[['feature_name' , 'OR', '95% CI', 'p_value']],
                on='feature_name', how='left', suffixes=['_UNI', '_MULTI']).applymap(lambda x: '' if pd.isna(x) else x)
info.applymap(lambda x: x if isinstance(x, str) or x > 0.05 else '<0.05')